# 02 Single Camera Data Pipeline

This notebook implements a PyTorch-compatible data loading pipeline for Project Aria Gen 1 recordings, treating `.vrs` files as indexed image sequences without extracting frames to disk.

### Background

The standard approach to loading video data in deep learning pipelines involves extracting frames as JPEG files before training. However, based on technical analysis, we adopted a **hybrid indexed-access approach** using `VrsDataProvider` directly, avoiding disk overhead while maintaining full random access compatibility with PyTorch's `DataLoader`.

### In this notebook, we will:

1. **Initialize the Data Provider:** Establish a connection with the `.vrs` file using the `projectaria_tools` library.

2. **RGB Stream Indexing:** Extract the full timestamp list for the **RGB camera** stream (`camera-rgb`, StreamId `214-1`) to build a random-access index map.

3. **AriaDataset Class:** Implement a PyTorch `Dataset` with `__len__` and `__getitem__`, supporting optional transforms and returning image tensors with their associated timestamps.

4. **DataLoader & Batch Test:** Instantiate the dataset, configure a `DataLoader` with a custom `collate_fn` for int64 timestamps, and iterate over the first batch.

5. **Multistream Extension Preview:** Demonstrate how the pipeline can be extended to additional streams (SLAM cameras, IMU) by adding StreamIds to `__getitem__`.

6. **Export `src/aria_dataset.py`:** Refactor the final class into a standalone reusable `.py` module.

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory.

For this notebook, we used as references:
* `projectaria_tools` documentation: https://facebookresearch.github.io/projectaria_tools/docs/intro 
* PyTorch `Dataset` / `DataLoader` documentation: https://pytorch.org/docs/stable/data.html

## 2.1 Initialize the Data Provider

The first step is to establish a connection with the `.vrs` file using the `VrsDataProvider` interface from `projectaria_tools`. 

In this notebook, we will work with `kettle_and_forklift_recording.vrs`, a recording that captures a kettle and a forklift. This file was already inspected and validated in Notebook 01, so it is an already reliable input for building the single-camera data loading pipeline.

In [4]:
import os
from projectaria_tools.core import data_provider

# Define paths to the .vrs file
DATA_DIR = os.path.join("..", "data", "raw", "kettle_and_forklift")
VRS_PATH = os.path.join(DATA_DIR, "kettle_and_forklift_recording.vrs")

print("VRS exists:", os.path.exists(VRS_PATH), "|", VRS_PATH)

VRS exists: True | ..\data\raw\kettle_and_forklift\kettle_and_forklift_recording.vrs


In [5]:
# Initialize the data provider for the .vrs file
if os.path.exists(VRS_PATH):
    provider = data_provider.create_vrs_data_provider(VRS_PATH)
    if provider:
        print("Data provider initialized successfully")
        print(f"Available streams: {len(provider.get_all_streams())}")
    else:
        print("Failed to initialize data provider")
else:
    provider = None
    print("VRS file not found")

Data provider initialized successfully
Available streams: 11


## 2.2 RGB Stream Indexing

To make the `.vrs` recording compatible with PyTorch-style random access, we first need to build an index for the RGB camera stream.

In practice, this means extracting the full list of timestamps for the RGB stream (`camera-rgb`, StreamId `214-1`) and storing them in a simple ordered structure. This timestamp list will act as the reference map used by the dataset: each integer index will correspond to one RGB frame and its associated device-time timestamp.

In [ ]:
from projectaria_tools.core import sensor_data

# Ensure the provider is available
if "provider" not in globals() or provider is None:
    raise RuntimeError("Provider is not initialized.")

# Define the expected RGB stream identifier (verified in Notebook 01)
RGB_STREAM_ID_STR = "214-1"

# Resolve the RGB stream object from its string identifier
rgb_candidates = [sid for sid in provider.get_all_streams() if str(sid) == RGB_STREAM_ID_STR]

if len(rgb_candidates) == 0:
    raise RuntimeError(f"RGB stream {RGB_STREAM_ID_STR} not found in the VRS file.")

rgb_stream_id = rgb_candidates[0] # Assuming the first match is the correct stream
rgb_label = provider.get_label_from_stream_id(rgb_stream_id) # get the stream label (Should be "camera-rgb")
rgb_sample_count = provider.get_num_data(rgb_stream_id) # Total number of samples in the RGB stream

print("RGB stream resolved successfully")
print("Stream ID   :", rgb_stream_id)
print("Stream label:", rgb_label)
print("Sample count:", rgb_sample_count)

RGB stream resolved successfully
Stream ID   : 214-1
Stream label: camera-rgb
Sample count: 1551


In [ ]:
import numpy as np
import pandas as pd

# Extract all device-time timestamps for the RGB stream
rgb_timestamps_ns = np.empty(rgb_sample_count, dtype=np.int64)

# Iterate through all samples in the RGB stream
for i in range(rgb_sample_count):
    # Read one RGB sample by index
    sample = provider.get_sensor_data_by_index(rgb_stream_id, i)

    # Store the device-time timestamp in nanoseconds
    rgb_timestamps_ns[i] = sample.get_time_ns(sensor_data.TimeDomain.DEVICE_TIME)

# Build a simple index table for inspection and debugging
# The DataFrame index corresponds to the dataset index used by the PyTorch dataset
rgb_index_df = pd.DataFrame(
    {"timestamp_ns": rgb_timestamps_ns},
    index=np.arange(rgb_sample_count, dtype=np.int64),
)

print(f"Extracted {len(rgb_timestamps_ns)} RGB timestamps.")

# Display the first 10 indexed timestamps
print("\nFirst 10 RGB frames (leftmost numbers are dataset indices):")
display(rgb_index_df.head(10))

# Display the last 10 indexed timestamps
print("\nLast 10 RGB frames (leftmost numbers are dataset indices):")
display(rgb_index_df.tail(10))

Extracted 1551 RGB timestamps.

First 10 RGB frames (leftmost numbers are dataset indices):


,timestamp_ns
0,888362615137
1,888395942475
2,888429273512
3,888462600850
4,888495914187
5,888529256600
6,888562584137
7,888595906475
8,888629247100
9,888662564562



Last 10 RGB frames (leftmost numbers are dataset indices):


,timestamp_ns
1541,939721060637
1542,939754387512
1543,939787716887
1544,939821049850
1545,939854370475
1546,939887701512
1547,939921031187
1548,939954362062
1549,939987683725
1550,940021010637


In [13]:
# Check timestamp monotonicity for the RGB stream
rgb_deltas_ns = np.diff(rgb_timestamps_ns)

print("Min delta [ms]   :", rgb_deltas_ns.min() / 1e6 if len(rgb_deltas_ns) > 0 else None)
print("Median delta [ms]:", np.median(rgb_deltas_ns) / 1e6 if len(rgb_deltas_ns) > 0 else None) # Median is more robust to outliers than mean
print("Max delta [ms]   :", rgb_deltas_ns.max() / 1e6 if len(rgb_deltas_ns) > 0 else None) 
print("Strictly increasing:", bool(np.all(rgb_deltas_ns > 0))) # True if all deltas are positive, indicating strictly increasing timestamps

Min delta [ms]   : 33.037587
Median delta [ms]: 33.327338
Max delta [ms]   : 33.6145
Strictly increasing: True
